In [16]:
import csv
import logging
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)

# Paths 
FORECAST_ROOT = Path('Data/Forecast')

# NWM column names 
COL_NWM_VERSION    = 'NWM_version_number'
COL_NWM_INIT_TIME  = 'model_initialization_time'
COL_NWM_VALID_TIME = 'model_output_valid_time'
COL_NWM_STREAMFLOW = 'streamflow_value'
COL_NWM_STREAM_ID  = 'streamID'
NWM_DATETIME_FMT   = '%Y-%m-%d_%H:%M:%S'

NWM_REQUIRED_COLS: frozenset[str] = frozenset({
    COL_NWM_VERSION, COL_NWM_INIT_TIME, COL_NWM_VALID_TIME,
    COL_NWM_STREAMFLOW, COL_NWM_STREAM_ID,
})

# USGS column names 
COL_USGS_DATETIME        = 'DateTime'
COL_USGS_FLOW            = 'USGSFlowValue'
COL_USGS_QUALITY         = '00060_cd'        
COL_USGS_QUALITY_LEGACY  = 'USGS_GageID'     


# Data structures 
@dataclass
class ForecastRow:
    """A single normalised forecast or observation record."""
    station_id:       Optional[str]
    source:           str                  
    forecast_init:    Optional[datetime]
    forecast_valid:   Optional[datetime]
    streamflow_value: Optional[float]
    nwm_version:      Optional[str]
    quality_code:     Optional[str]


# Parsing helpers 
def try_float(value: Optional[str]) -> Optional[float]:
    """Return *value* as a float, or None if blank/unparseable."""
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def parse_nwm_datetime(value: Optional[str]) -> Optional[datetime]:
    """Parse an NWM datetime string (``YYYY-MM-DD_HH:MM:SS``)."""
    if not value:
        return None
    try:
        return datetime.strptime(value, NWM_DATETIME_FMT)
    except ValueError:
        log.warning("Could not parse NWM datetime: %r", value)
        return None


def parse_usgs_datetime(value: Optional[str]) -> Optional[datetime]:
    """Parse a USGS datetime string, stripping timezone info when present."""
    if not value:
        return None
    try:
        dt = datetime.fromisoformat(value)
        return dt.replace(tzinfo=None)
    except ValueError:
        pass
    try:
        return datetime.strptime(value, '%Y-%m-%d %H:%M:%S%z').replace(tzinfo=None)
    except ValueError:
        log.warning("Could not parse USGS datetime: %r", value)
        return None


# CSV loading 
def _detect_format(headers: list[str]) -> str:
    """Return ``'NWM'``, ``'USGS'``, or ``'unknown'`` based on column headers."""
    header_set = set(headers)
    if NWM_REQUIRED_COLS.issubset(header_set):
        return 'NWM'
    if COL_USGS_DATETIME in header_set:
        return 'USGS'
    return 'unknown'


def _parse_nwm_row(raw: dict[str, str], station_id: str) -> ForecastRow:
    return ForecastRow(
        station_id=station_id,
        source='NWM',
        forecast_init=parse_nwm_datetime(raw.get(COL_NWM_INIT_TIME, '')),
        forecast_valid=parse_nwm_datetime(raw.get(COL_NWM_VALID_TIME, '')),
        streamflow_value=try_float(raw.get(COL_NWM_STREAMFLOW, '')),
        nwm_version=raw.get(COL_NWM_VERSION, ''),
        quality_code=None,
    )


def _parse_usgs_row(raw: dict[str, str], station_id: str) -> ForecastRow:
    ts = parse_usgs_datetime(raw.get(COL_USGS_DATETIME, ''))
    quality = raw.get(COL_USGS_QUALITY) or raw.get(COL_USGS_QUALITY_LEGACY, '')
    return ForecastRow(
        station_id=station_id,
        source='USGS',
        forecast_init=ts,
        forecast_valid=ts,
        streamflow_value=try_float(raw.get(COL_USGS_FLOW, '')),
        nwm_version=None,
        quality_code=quality,
    )


def read_forecast_csv(path: Path) -> list[ForecastRow]:
    """Read one forecast CSV and return its rows as :class:`ForecastRow` objects."""
    rows: list[ForecastRow] = []
    with path.open(newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            log.warning("Empty or header-less file: %s", path.name)
            return rows

        headers = [h.strip() for h in reader.fieldnames]
        fmt = _detect_format(headers)
        if fmt == 'unknown':
            log.warning("Unrecognised format in %s — skipping", path.name)
            return rows

        # USGS station id is encoded in the filename as '<station_id>_*.csv'
        station_id = path.stem.split('_', 1)[0]

        for raw in reader:
            if fmt == 'NWM':
                rows.append(_parse_nwm_row(raw, station_id))
            else:
                rows.append(_parse_usgs_row(raw, station_id))

    return rows


def load_station_data(station_path: Path) -> list[ForecastRow]:
    """Load and time-sort all CSVs found directly under *station_path*."""
    rows: list[ForecastRow] = []
    for csv_file in sorted(station_path.glob('*.csv')):
        rows.extend(read_forecast_csv(csv_file))

    rows.sort(key=lambda r: (
        r.forecast_valid or datetime.min,
        r.forecast_init  or datetime.min,
    ))
    return rows


# Load all stations 
station_directories = sorted(p for p in FORECAST_ROOT.iterdir() if p.is_dir())
station_data: dict[str, list[ForecastRow]] = {
    d.name: load_station_data(d) for d in station_directories
}


In [17]:
@dataclass
class CleanSummary:
    """Per-station statistics after dropping rows with missing streamflow."""
    total_rows:   int
    cleaned_rows: int
    dropped_rows: int
    drop_rate:    float


def clean_station_data(
    raw: dict[str, list[ForecastRow]],
) -> tuple[dict[str, list[ForecastRow]], dict[str, CleanSummary]]:
    """Drop rows missing a streamflow value and return per-station summaries."""
    cleaned: dict[str, list[ForecastRow]] = {}
    summaries: dict[str, CleanSummary] = {}

    for station_name, rows in raw.items():
        valid    = [r for r in rows if r.streamflow_value is not None]
        dropped  = len(rows) - len(valid)
        cleaned[station_name]  = valid
        summaries[station_name] = CleanSummary(
            total_rows=len(rows),
            cleaned_rows=len(valid),
            dropped_rows=dropped,
            drop_rate=dropped / len(rows) if rows else 0.0,
        )

    return cleaned, summaries


cleaned_station_data, station_summaries = clean_station_data(station_data)

print('Cleaned summary by station:')
for station_name, s in station_summaries.items():
    print(
        f'  {station_name}: '
        f'total={s.total_rows}, '
        f'cleaned={s.cleaned_rows}, '
        f'dropped={s.dropped_rows} ({s.drop_rate:.1%})'
    )


Cleaned summary by station:
  20380357: total=396249, cleaned=396248, dropped=1 (0.0%)
  21609641: total=391658, cleaned=391643, dropped=15 (0.0%)


In [18]:
import pandas as pd
from dataclasses import asdict


def _quality_is_accepted(code: Optional[str]) -> bool:
    """Return False only when the USGS quality string contains the estimated flag 'e'."""
    if not code:
        return True   
    return 'e' not in code.split()


def align_station(rows: list[ForecastRow]) -> pd.DataFrame:
    """Join NWM forecasts to USGS observations for one station.

    Steps:
      1. Filter USGS rows to accepted quality codes (drop estimated 'e' readings).
      2. Resample USGS to hourly mean (USGS is 15-min; NWM is hourly).
      3. Compute NWM lead_hours = forecast_valid − forecast_init.
      4. Inner-join NWM rows to hourly USGS index on forecast_valid.

    Returns a DataFrame with columns:
        forecast_valid, forecast_init, nwm_streamflow,
        usgs_streamflow, lead_hours, station_id
    """
    df = pd.DataFrame([asdict(r) for r in rows])

    nwm  = df[df['source'] == 'NWM'].copy()
    usgs = df[df['source'] == 'USGS'].copy()

    if nwm.empty or usgs.empty:
        return pd.DataFrame()

    # USGS: quality filter → resample to hourly 
    usgs = usgs[usgs['quality_code'].apply(_quality_is_accepted)]
    if usgs.empty:
        return pd.DataFrame()

    usgs['forecast_valid'] = pd.to_datetime(usgs['forecast_valid'])
    usgs_hourly = (
        usgs.set_index('forecast_valid')['streamflow_value']
        .resample('1h').mean()
        .rename('usgs_streamflow')
        .dropna()
    )

    # NWM: compute lead time 
    nwm['forecast_valid'] = pd.to_datetime(nwm['forecast_valid'])
    nwm['forecast_init']  = pd.to_datetime(nwm['forecast_init'])
    nwm['lead_hours'] = (
        (nwm['forecast_valid'] - nwm['forecast_init'])
        .dt.total_seconds()
        .div(3600)
        .round()
        .astype(int)
    )
    nwm = nwm.rename(columns={'streamflow_value': 'nwm_streamflow'})

    # Inner join on forecast_valid timestamp 
    aligned = (
        nwm.set_index('forecast_valid')
        .join(usgs_hourly, how='inner')
        .reset_index()
    )

    keep = ['forecast_valid', 'forecast_init', 'nwm_streamflow',
            'usgs_streamflow', 'lead_hours', 'station_id']
    return aligned[keep].dropna()


aligned_data: dict[str, pd.DataFrame] = {}

for station_name, rows in cleaned_station_data.items():
    df = align_station(rows)
    if df.empty:
        print(f'{station_name}: no aligned pairs found')
        continue
    aligned_data[station_name] = df
    print(
        f'{station_name}: {len(df):,} aligned pairs  '
        f'lead_hours {df["lead_hours"].min()}–{df["lead_hours"].max()}  '
        f'({df["forecast_valid"].min().date()} → {df["forecast_valid"].max().date()})'
    )

print(f'\nTotal across all stations: {sum(len(v) for v in aligned_data.values()):,} paired rows')


20380357: 324,904 aligned pairs  lead_hours 1–18  (2021-04-21 → 2023-04-22)
21609641: 302,406 aligned pairs  lead_hours 1–18  (2021-04-21 → 2023-04-22)

Total across all stations: 627,310 paired rows


In [19]:
# Season index: 0=winter, 1=spring, 2=summer, 3=autumn
SEASON_MAP: dict[int, int] = {
    12: 0,  1: 0,  2: 0,
     3: 1,  4: 1,  5: 1,
     6: 2,  7: 2,  8: 2,
     9: 3, 10: 3, 11: 3,
}

FEATURE_COLS = ['nwm_streamflow', 'lead_hours', 'hour_of_day', 'month', 'season']
TARGET_COL   = 'usgs_streamflow'


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Append hour_of_day, month, and season columns derived from forecast_valid."""
    dt = pd.to_datetime(df['forecast_valid'])
    return df.assign(
        hour_of_day=dt.dt.hour,
        month=dt.dt.month,
        season=dt.dt.month.map(SEASON_MAP),
    )


featured_data: dict[str, pd.DataFrame] = {
    name: add_temporal_features(df.copy())
    for name, df in aligned_data.items()
}


In [20]:
from sklearn.preprocessing import StandardScaler

# Project-specified time boundaries (Section 3 of project description)
TRAIN_END  = pd.Timestamp('2022-06-30 23:00:00')  # ~80% of the Apr 2021–Sep 2022 window
VAL_END    = pd.Timestamp('2022-09-30 23:00:00')   # remainder of allowed train/val window
TEST_START = pd.Timestamp('2022-10-01 00:00:00')   # strictly held-out test period

scalers: dict[str, StandardScaler] = {}
splits:  dict[str, dict]           = {}

for station_name, df in featured_data.items():
    df = df.sort_values('forecast_valid').reset_index(drop=True)
    fv = pd.to_datetime(df['forecast_valid'])

    train = df[fv <= TRAIN_END].copy()
    val   = df[(fv > TRAIN_END) & (fv <= VAL_END)].copy()
    test  = df[fv >= TEST_START].copy()

    # Fit scaler ONLY on training data — transform val/test with the same scaler
    scaler = StandardScaler()
    train[['nwm_streamflow', TARGET_COL]] = scaler.fit_transform(
        train[['nwm_streamflow', TARGET_COL]]
    )
    val[['nwm_streamflow', TARGET_COL]] = scaler.transform(
        val[['nwm_streamflow', TARGET_COL]]
    )
    test[['nwm_streamflow', TARGET_COL]] = scaler.transform(
        test[['nwm_streamflow', TARGET_COL]]
    )

    scalers[station_name] = scaler
    splits[station_name]  = {'train': train, 'val': val, 'test': test, 'scaler': scaler}

    print(f'{station_name}:')
    print(f'  train {len(train):>7,}  {train["forecast_valid"].min().date()} → {train["forecast_valid"].max().date()}')
    print(f'  val   {len(val):>7,}  {val["forecast_valid"].min().date()} → {val["forecast_valid"].max().date()}')
    print(f'  test  {len(test):>7,}  {test["forecast_valid"].min().date()} → {test["forecast_valid"].max().date()}')


20380357:
  train 194,229  2021-04-21 → 2022-06-30
  val    39,850  2022-07-01 → 2022-09-30
  test   90,825  2022-10-01 → 2023-04-22
21609641:
  train 170,541  2021-04-21 → 2022-06-30
  val    41,040  2022-07-01 → 2022-09-30
  test   90,825  2022-10-01 → 2023-04-22


In [21]:
import numpy as np

SEQ_LEN = 18  # one complete NWM forecast cycle: lead hours 1–18

def build_sequences(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Group rows by forecast_init and stack into fixed-length sequences.

    Returns:
        X   – (N, 18, n_features)  model inputs
        y   – (N, 18)              USGS streamflow targets
        nwm – (N, 18)              raw NWM forecast (baseline for evaluation)
    """
    X_list, y_list, nwm_list = [], [], []
    for _, group in df.groupby('forecast_init'):
        group = group.sort_values('lead_hours')
        if len(group) != SEQ_LEN:
            continue  # skip incomplete forecast cycles
        X_list.append(group[FEATURE_COLS].to_numpy(dtype=np.float32))
        y_list.append(group[TARGET_COL].to_numpy(dtype=np.float32))
        nwm_list.append(group['nwm_streamflow'].to_numpy(dtype=np.float32))

    return (
        np.stack(X_list),    # (N, 18, 5)
        np.stack(y_list),    # (N, 18)
        np.stack(nwm_list),  # (N, 18)
    )


seq_data: dict[str, dict] = {}
for station_name, s in splits.items():
    X_tr, y_tr, nwm_tr = build_sequences(s['train'])
    X_va, y_va, nwm_va = build_sequences(s['val'])
    X_te, y_te, nwm_te = build_sequences(s['test'])

    seq_data[station_name] = {
        'X_train': X_tr, 'y_train': y_tr, 'nwm_train': nwm_tr,
        'X_val':   X_va, 'y_val':   y_va, 'nwm_val':   nwm_va,
        'X_test':  X_te, 'y_test':  y_te, 'nwm_test':  nwm_te,
        'scaler':  s['scaler'],
    }
    print(f'{station_name}:  train={X_tr.shape}  val={X_va.shape}  test={X_te.shape}')


20380357:  train=(10110, 18, 5)  val=(2054, 18, 5)  test=(4690, 18, 5)
21609641:  train=(8629, 18, 5)  val=(2119, 18, 5)  test=(4690, 18, 5)


In [22]:
import tensorflow as tf
from tensorflow import keras

def build_lstm(seq_len: int, n_features: int,
               hidden: int = 64, dropout: float = 0.2) -> keras.Model:
    """Stacked LSTM that corrects NWM forecasts at every lead time simultaneously."""
    inp = keras.Input(shape=(seq_len, n_features), name='forecast_sequence')
    x   = keras.layers.LSTM(hidden, return_sequences=True)(inp)
    x   = keras.layers.Dropout(dropout)(x)
    x   = keras.layers.LSTM(hidden // 2, return_sequences=True)(x)
    x   = keras.layers.Dropout(dropout)(x)
    out = keras.layers.TimeDistributed(keras.layers.Dense(1))(x)
    out = keras.layers.Reshape((seq_len,), name='corrected_streamflow')(out)

    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return model


SEQ_LEN    = 18
N_FEATURES = len(FEATURE_COLS)

models: dict[str, keras.Model] = {
    name: build_lstm(SEQ_LEN, N_FEATURES) for name in seq_data
}
models[next(iter(models))].summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ forecast_sequence (InputLayer)  │ (None, 18, 5)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 18, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 18, 32)         │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 18, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 18, 1)          │            33 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ corrected_streamflow (Reshape)  │ (None, 18)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
EPOCHS     = 50
BATCH_SIZE = 256

histories: dict = {}

for station_name, sd in seq_data.items():
    print(f'\n── Training station {station_name} ──')
    model = models[station_name]

    history = model.fit(
        sd['X_train'], sd['y_train'],
        validation_data=(sd['X_val'], sd['y_val']),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5,
                restore_best_weights=True, verbose=1,
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3, verbose=1,
            ),
        ],
        verbose=1,
    )
    histories[station_name] = history



── Training station 20380357 ──
Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 1.0266 - val_loss: 0.7823 - learning_rate: 0.0010
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.9993 - val_loss: 0.8105 - learning_rate: 0.0010
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.9904 - val_loss: 0.6687 - learning_rate: 0.0010
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.9723 - val_loss: 0.5732 - learning_rate: 0.0010
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.9568 - val_loss: 0.5239 - learning_rate: 0.0010
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.9513 - val_loss: 0.5579 - learning_rate: 0.0010
Epoch 7/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.9489 - val_loss: 0.5343 - learning_rate: 0.0010
Epoch 8/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.9959
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.9401 - val_l

In [24]:
import numpy as np

def nse(obs: np.ndarray, sim: np.ndarray) -> float:
    """Nash-Sutcliffe Efficiency. Perfect = 1.0; climatological mean benchmark = 0.0."""
    return float(1.0 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2))

def rmse(obs: np.ndarray, sim: np.ndarray) -> float:
    return float(np.sqrt(np.mean((obs - sim) ** 2)))

def mae_score(obs: np.ndarray, sim: np.ndarray) -> float:
    return float(np.mean(np.abs(obs - sim)))


def _inv_col(scaled: np.ndarray, scaler: StandardScaler, col: int) -> np.ndarray:
    """Inverse-transform one column of a two-column StandardScaler."""
    flat  = scaled.flatten()
    dummy = np.zeros((len(flat), 2), dtype=np.float64)
    dummy[:, col] = flat
    return scaler.inverse_transform(dummy)[:, col].reshape(scaled.shape)


for station_name, sd in seq_data.items():
    scaler = sd['scaler']

    y_pred_scaled = models[station_name].predict(sd['X_test'], verbose=0)  # (N, 18)

    # Inverse-transform to physical units
    # col 0 = nwm_streamflow, col 1 = usgs_streamflow (scaler fit order)
    obs  = _inv_col(sd['y_test'],      scaler, col=1)  # (N, 18)
    pred = _inv_col(y_pred_scaled,      scaler, col=1)  # (N, 18)
    nwm  = _inv_col(sd['nwm_test'],     scaler, col=0)  # (N, 18)

    obs_f, pred_f, nwm_f = obs.flatten(), pred.flatten(), nwm.flatten()

    print(f'\n{station_name} — Test Set (Oct 2022 – Apr 2023)')
    print(f'  {"Model":22s}  {"NSE":>7}  {"RMSE":>8}  {"MAE":>8}')
    print(f'  {"LSTM (corrected)":22s}  {nse(obs_f,  pred_f):>7.4f}  {rmse(obs_f,  pred_f):>8.4f}  {mae_score(obs_f,  pred_f):>8.4f}')
    print(f'  {"NWM (original)":22s}  {nse(obs_f,  nwm_f):>7.4f}  {rmse(obs_f,  nwm_f):>8.4f}  {mae_score(obs_f,  nwm_f):>8.4f}')

    print(f'\n  Per-lead-hour NSE:')
    print(f'  {"Lead":>5}  {"LSTM":>8}  {"NWM":>8}')
    for lead in range(SEQ_LEN):
        print(f'  {lead+1:>4}h  {nse(obs[:, lead], pred[:, lead]):>8.4f}  {nse(obs[:, lead], nwm[:, lead]):>8.4f}')



20380357 — Test Set (Oct 2022 – Apr 2023)
  Model                       NSE      RMSE       MAE
  LSTM (corrected)        -0.1866    0.1909    0.1125
  NWM (original)          -206892.4296   79.7317   58.0047

  Per-lead-hour NSE:
   Lead      LSTM       NWM
     1h   -0.0266  -2341.5859
     2h   -0.0454  -9975.6658
     3h   -0.0679  -28233.0033
     4h   -0.0920  -60144.2980
     5h   -0.1166  -103435.6565
     6h   -0.1415  -148826.5064
     7h   -0.1663  -190508.3396
     8h   -0.1901  -227197.9949
     9h   -0.2119  -255083.4882
    10h   -0.2305  -274598.8831
    11h   -0.2451  -287782.1642
    12h   -0.2551  -296309.9515
    13h   -0.2611  -301658.7911
    14h   -0.2638  -304938.4020
    15h   -0.2639  -306913.8672
    16h   -0.2626  -308064.3063
    17h   -0.2604  -308720.3980
    18h   -0.2580  -309073.2609

21609641 — Test Set (Oct 2022 – Apr 2023)
  Model                       NSE      RMSE       MAE
  LSTM (corrected)         0.8580    6.2649    2.6037
  NWM (original)   